# 04 — Alvo supervisionado e funcionamento do BERTimbau

Este notebook traduz os requisitos da Sprint 3 para um experimento concreto e mostra como o BERTimbau receberá os trechos das reuniões.

## Contrato da Sprint 3

A Sprint permite diferentes problemas de classificação. Para manter um experimento claro e alinhado à base comercial TOTVS, adotamos inicialmente:

- **Problema:** detecção de oportunidade comercial;
- **Classes:** `nao_oportunidade` e `oportunidade`;
- **Unidade rotulada:** chunk de transcrição;
- **Unidade de separação:** reunião (`meeting_id`);
- **Agregação futura:** maior probabilidade de oportunidade entre os chunks da reunião, com limiar validado;
- **Algoritmo 1:** TF-IDF + Logistic Regression;
- **Algoritmo 2:** BERTimbau fine-tuned para classificação binária;
- **Métricas:** confusion matrix, precision, recall, F1-score e accuracy;
- **Métrica principal proposta:** recall da classe `oportunidade`, acompanhado de precision e F1. Perder uma oportunidade real pode ser mais caro do que revisar um alerta falso, mas o limiar deverá controlar excesso de alertas.

Os dois modelos usarão os mesmos rótulos e o mesmo conjunto de teste agrupado por reunião.

## Guia inicial de anotação

### `oportunidade`
Marcar quando o trecho trouxer evidência comercial acionável, por exemplo: dor que pode ser atendida, necessidade explícita, intenção de compra, avaliação ou troca de fornecedor, pedido de proposta/demonstração, orçamento, prazo de decisão, expansão, cross-sell ou upsell.

### `nao_oportunidade`
Marcar quando o trecho for contextual, operacional ou social sem necessidade comercial acionável: apresentação, confirmação de agenda, suporte rotineiro, descrição neutra do ambiente ou menção de produto sem dor/intenção.

### `revisao`
Usar somente no processo de anotação quando faltar contexto ou houver dúvida. Esses exemplos não entram no treino até revisão humana. `revisao` não é uma terceira classe do modelo.

A base RAG pode ajudar o anotador a compreender produtos e dores, mas nunca deve gerar o rótulo de verdade automaticamente.

## Como o BERTimbau será usado

O BERTimbau disponível hoje é um modelo de linguagem pré-treinado, não um classificador comercial pronto. O fine-tuning adiciona uma cabeça de classificação sobre a representação do token `[CLS]` e ajusta os pesos usando nossos chunks rotulados.

Fluxo de um chunk:

1. o tokenizer converte texto em IDs de subpalavras;
2. adiciona `[CLS]` no início e `[SEP]` no fim;
3. aplica padding e attention mask até o tamanho configurado;
4. o BERTimbau produz uma representação contextual;
5. a cabeça de classificação retorna logits para as duas classes;
6. softmax converte logits em probabilidades;
7. o limiar de decisão é escolhido na validação, nunca no teste.

Sem rótulos humanos, executar o modelo pré-treinado permitiria apenas tarefas como preencher palavras mascaradas; isso não mede oportunidade comercial.

In [ ]:
from pathlib import Path
import json

import transformers
from transformers import AutoConfig, AutoTokenizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CHUNKS_PATH = PROJECT_ROOT / 'data' / 'processed' / 'chunks_bertimbau.jsonl'
CHECKPOINT = 'neuralmind/bert-base-portuguese-cased'
LABEL2ID = {'nao_oportunidade': 0, 'oportunidade': 1}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}

C:\Users\Gabriel Pereira\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Verificar o contrato atual dos dados

A inspeção mostra apenas os nomes dos campos. Nenhum conteúdo de transcrição é exibido.

In [2]:
with CHUNKS_PATH.open('r', encoding='utf-8') as source:
    first_chunk = json.loads(next(source))
fields = sorted(first_chunk)
label_like_fields = [field for field in fields if any(term in field.casefold() for term in ('label', 'rotulo', 'rótulo', 'target', 'classe'))]
{'available_fields': fields, 'existing_label_fields': label_like_fields}

{'available_fields': ['CNAE',
  'CODT',
  'DT_CRIACAO',
  'DT_MEETING',
  'DURACAO_MEETING',
  'FAIXA_FATURAMENTO_CLIENTE_EC',
  'FLG_EXTERNO',
  'FORMATO_MEETING',
  'ID_MEETING',
  'ID_STATUS_MEETING',
  'NOME_SEGMENTO',
  'NOME_UNIDADE',
  'NUM_CARACTERES_TRANSCRICAO',
  'NUM_TURNOS_TRANSCRICAO',
  'STATUS_MEETING',
  'UF',
  'char_end',
  'char_start',
  'checkpoint',
  'chunk_id',
  'chunk_index',
  'content_max_tokens',
  'meeting_id',
  'num_tokens',
  'overlap_tokens_target',
  'speakers',
  'text',
  'turn_end',
  'turn_start'],
 'existing_label_fields': ['overlap_tokens_target']}

## Demonstração segura do tokenizer

Usamos frases sintéticas para visualizar as entradas. O mesmo processo será aplicado aos chunks reais sem exibi-los.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=True, do_lower_case=False)
examples = [
    'Precisamos automatizar as aprovações e gostaríamos de avaliar uma solução.',
    'A reunião de acompanhamento ficou marcada para terça-feira.',
]
encoded = tokenizer(examples, padding=True, truncation=True, max_length=512)
[
    {
        'synthetic_text': text,
        'tokens': tokenizer.convert_ids_to_tokens(input_ids),
        'attention_tokens': sum(mask),
    }
    for text, input_ids, mask in zip(examples, encoded['input_ids'], encoded['attention_mask'])
]

[{'synthetic_text': 'Precisamos automatizar as aprovações e gostaríamos de avaliar uma solução.',
  'tokens': ['[CLS]',
   'Pre',
   '##cis',
   '##amos',
   'automa',
   '##tizar',
   'as',
   'aprov',
   '##ações',
   'e',
   'gosta',
   '##rí',
   '##amos',
   'de',
   'avaliar',
   'uma',
   'solução',
   '.',
   '[SEP]'],
  'attention_tokens': 19},
 {'synthetic_text': 'A reunião de acompanhamento ficou marcada para terça-feira.',
  'tokens': ['[CLS]',
   'A',
   'reunião',
   'de',
   'acompanhamento',
   'ficou',
   'marcada',
   'para',
   'ter',
   '##ça',
   '-',
   'feira',
   '.',
   '[SEP]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]',
   '[PAD]'],
  'attention_tokens': 14}]

## Configuração futura do classificador

Esta célula prepara a configuração da cabeça binária sem baixar os pesos do modelo. O fine-tuning exigirá PyTorch e o conjunto anotado.

In [4]:
config = AutoConfig.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABEL2ID),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)
{
    'model_type': config.model_type,
    'hidden_size': config.hidden_size,
    'layers': config.num_hidden_layers,
    'attention_heads': config.num_attention_heads,
    'max_position_embeddings': config.max_position_embeddings,
    'labels': config.id2label,
    'transformers_version': transformers.__version__,
}

{'model_type': 'bert',
 'hidden_size': 768,
 'layers': 12,
 'attention_heads': 12,
 'max_position_embeddings': 512,
 'labels': {0: 'nao_oportunidade', 1: 'oportunidade'},
 'transformers_version': '5.17.0'}

## Próxima etapa necessária

Antes de treinar, precisamos criar uma fila de anotação e obter rótulos humanos. Depois construiremos uma única divisão por `meeting_id`, treinaremos Logistic Regression e BERTimbau nela e compararemos as métricas no mesmo teste.